# 01 - Setup and governance
Validates the CUAD staging, synthetic overlay and Unity Catalog governance (masks, row filter, tags) against the live workspace.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from dbx import sql, ws, CATALOG
print('workspace:', ws().config.host, '| catalog:', CATALOG, '| user:', ws().current_user.me().user_name)

workspace: https://fevm-serverless-stable-oj7eie.cloud.databricks.com | catalog: serverless_stable_oj7eie_catalog | user: erica.yuen@databricks.com


In [2]:
sql("SELECT file_name, cuad_category FROM ${catalog}.raw.cuad_manifest ORDER BY cuad_category, file_name")

,file_name,cuad_category
0,GULFSOUTHMEDICALSUPPLYINC_12_24_1997-EX-4-AFFILIATE AGREEMENT.PDF,Affiliate Agreement
1,CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement.pdf,Affiliate_Agreements
2,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf,Affiliate_Agreements
3,DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.pdf,Affiliate_Agreements
4,LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.pdf,Affiliate_Agreements
...,...,...
95,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_EX-10.16_Supply Agreement.pdf,Supply
96,ReynoldsConsumerProductsInc_20191115_S-1_EX-10.18_11896469_EX-10.18_Supply Agreement.pdf,Supply
97,PenntexMidstreamPartnersLp_20150416_S-1A_EX-10.4_9042833_EX-10.4_Transportation Agreement.pdf,Transportation
98,RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_9045501_EX-10.5_Transportation Agreement.pdf,Transportation


In [3]:
sql("LIST '/Volumes/${catalog}/raw/contracts/pdfs/'")[['name','size']].head(25)

,name,size
0,2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.pdf,109658
1,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT.PDF,47559
2,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT.PDF,125139
3,"ADUROBIOTECH,INC_06_02_2020-EX-10.7-CONSULTING AGREEMENT(1).PDF",98410
4,"ADUROBIOTECH,INC_06_02_2020-EX-10.7-CONSULTING AGREEMENT.PDF",98410
5,ALLISONTRANSMISSIONHOLDINGSINC_12_15_2014-EX-99.1-COOPERATION AGREEMENT.PDF,139128
6,AMBASSADOREYEWEARGROUPINC_11_17_1997-EX-10.28-ENDORSEMENT AGREEMENT.PDF,71470
7,ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT.PDF,93991
8,"ATHENSBANCSHARESCORP_11_02_2009-EX-1.2-AGENCY AGREEMENT , 2009.PDF",179058
9,AgapeAtpCorp_20191202_10-KA_EX-10.1_11911128_EX-10.1_Supply Agreement.pdf,178486


In [4]:
sql("LIST '/Volumes/${catalog}/raw/contracts/labels/'")[['name','size']]

,name,size
0,CUAD_v1.json,40128638
1,CUAD_v1_README.txt,26712
2,master_clauses.csv,3955428


In [5]:
sql("SELECT * FROM ${catalog}.curated.matters ORDER BY matter_seq")

,matter_seq,matter_id,client,practice_area,matter_team
0,0,M-1001,"Rodriguez, Figueroa and Sanchez (fictional)",Commercial Contracts,Team Atlas
1,1,M-1002,Doyle Ltd (fictional),Technology Transactions,Team Borealis
2,2,M-1003,"Mcclain, Miller and Henderson (fictional)",IP Licensing,Team Cascade
3,3,M-1004,Davis and Sons (fictional),Supply Chain,Team Denali
4,4,M-1005,"Guzman, Hoffman and Baldwin (fictional)",Real Estate,Team Atlas
5,5,M-1006,"Gardner, Robinson and Lawrence (fictional)",Employment,Team Borealis
6,6,M-1007,Blake and Sons (fictional),Commercial Contracts,Team Cascade
7,7,M-1008,"Henderson, Ramirez and Lewis (fictional)",Technology Transactions,Team Denali
8,8,M-1009,Garcia-James (fictional),IP Licensing,Team Atlas
9,9,M-1010,Abbott-Munoz (fictional),Supply Chain,Team Borealis


In [6]:
sql("SELECT * FROM ${catalog}.curated.user_team_mapping ORDER BY matter_team")

,user_email,matter_team
0,arnoldmaria@example.com,Team Atlas
1,erica.yuen@databricks.com,Team Atlas
2,xreid@example.com,Team Atlas
3,erica.yuen@databricks.com,Team Borealis
4,mitchellclark@example.com,Team Borealis
5,barbara10@example.com,Team Borealis
6,kendragalloway@example.com,Team Cascade
7,jacqueline19@example.com,Team Cascade
8,michellejames@example.com,Team Denali
9,amandasanchez@example.com,Team Denali


In [7]:
sql("SELECT * FROM ${catalog}.curated.risk_playbook ORDER BY rule_id")

,rule_id,clause_type,rule_type,match_value,severity,rule_text
0,R01,uncapped_liability,is_true,NaN,high,Liability is uncapped or unlimited for at least one party.
1,R02,non_compete,present,NaN,high,Contract restricts a party from competing; requires partner sign-off.
2,R03,auto_renewal,is_true,NaN,medium,Contract renews automatically unless notice is given; calendar the notice date.
3,R04,notice_period_days,gte,90,medium,Notice period to stop renewal is 90 days or longer.
4,R05,exclusivity,present,NaN,medium,Contract grants or requires exclusivity.
5,R06,governing_law,not_in,"delaware,new york,california",low,"Governing law is outside the firm's preferred jurisdictions (DE, NY, CA)."
6,R07,termination_for_convenience,present,NaN,low,A party may terminate for convenience (without cause).
7,R08,assignment_restriction,present,NaN,low,Assignment requires consent or is restricted (change-of-control risk).


In [8]:
sql("SELECT section, status, statement, error FROM ${catalog}.curated.governance_log ORDER BY ts DESC LIMIT 20")

,section,status,statement,error
0,post,allowed_failure,"GRANT USE SCHEMA, SELECT ON SCHEMA serverless_stable_oj7eie_catalog.curated TO `legal_reviewers_full`",[RequestId=4b60f136-03ca-4775-af54-a1aab7fbe26a ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could ...
1,post,allowed_failure,GRANT USE CATALOG ON CATALOG serverless_stable_oj7eie_catalog TO `legal_reviewers_full`,[RequestId=e14207e9-6c7c-47b4-a1cb-73af5bcc41f6 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could ...
2,post,allowed_failure,"GRANT USE SCHEMA, SELECT ON SCHEMA serverless_stable_oj7eie_catalog.curated TO `legal_reviewers`",[RequestId=8ee839bf-4c13-4fc9-bc5a-415d8d3141e4 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could ...
3,post,allowed_failure,GRANT USE CATALOG ON CATALOG serverless_stable_oj7eie_catalog TO `legal_reviewers`,[RequestId=6d439937-5253-46aa-9caf-dcf1f6784e7f ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could ...
4,post,ok,ALTER MATERIALIZED VIEW serverless_stable_oj7eie_catalog.curated.gold_risk_flags SET TAGS ('privileged' = 'true'),NaN
5,post,ok,ALTER MATERIALIZED VIEW serverless_stable_oj7eie_catalog.curated.gold_documents SET TAGS ('privileged' = 'true'),NaN
6,post,ok,ALTER MATERIALIZED VIEW serverless_stable_oj7eie_catalog.curated.gold_clauses SET TAGS ('privileged' = 'true'),NaN
7,post,ok,ALTER STREAMING TABLE serverless_stable_oj7eie_catalog.curated.silver_parsed ALTER COLUMN full_text SET TAGS ('data_...,NaN
8,post,ok,ALTER MATERIALIZED VIEW serverless_stable_oj7eie_catalog.curated.gold_documents ALTER COLUMN party_name SET TAGS ('d...,NaN
9,post,ok,ALTER MATERIALIZED VIEW serverless_stable_oj7eie_catalog.curated.gold_clauses ALTER COLUMN party_name SET TAGS ('dat...,NaN


In [9]:
sql("SHOW USER FUNCTIONS IN ${catalog}.curated")

,function
0,serverless_stable_oj7eie_catalog.curated.mask_parties_clause
1,serverless_stable_oj7eie_catalog.curated.mask_party
2,serverless_stable_oj7eie_catalog.curated.matter_team_filter


In [10]:
d = sql("DESCRIBE TABLE EXTENDED ${catalog}.curated.gold_clauses")
d[d['col_name'].str.contains('Mask|Row Filter|party_name|extracted_value|Tags', case=False, regex=True)]

,col_name,data_type,comment
3,extracted_value,string,NaN
9,party_name,string,NaN
35,Row Filter,`serverless_stable_oj7eie_catalog`.`curated`.`matter_team_filter` ON (matter_team),
37,# Column Masks,,
38,extracted_value,`serverless_stable_oj7eie_catalog`.`curated`.`mask_parties_clause`,USING COLUMNS(clause_type)
41,party_name,`serverless_stable_oj7eie_catalog`.`curated`.`mask_party`,


In [11]:
sql("SELECT schema_name, table_name, column_name, tag_name, tag_value FROM ${catalog}.information_schema.column_tags")

,schema_name,table_name,column_name,tag_name,tag_value
0,curated,silver_parsed,full_text,data_classification,pii
1,curated,gold_documents,party_name,data_classification,pii
2,curated,gold_clauses,party_name,data_classification,pii


In [12]:
sql("SELECT schema_name, tag_name, tag_value FROM ${catalog}.information_schema.schema_tags")

,schema_name,tag_name,tag_value
0,curated,privileged,true


In [13]:
# As the current (admin) user: all matter teams visible, party names unmasked
sql("SELECT current_user() u, is_member('legal_admins') admin, is_member('legal_reviewers_full') full_reviewer")

,u,admin,full_reviewer
0,erica.yuen@databricks.com,true,true


In [14]:
sql("SELECT matter_team, count(*) docs, min(party_name) sample_party FROM ${catalog}.curated.gold_documents GROUP BY 1 ORDER BY 1")

,matter_team,docs,sample_party
0,Team Atlas,25,"ALPS Distributors, Inc.; S2K Financial LLC"
1,Team Borealis,23,"Aduro Biotech, Inc.;IREYA B.V"
2,Team Cascade,20,ARCONIC INC.; ARCONIC ROLLED PRODUCTS CORP.
3,Team Denali,32,"ARCA U.S. TREASURY FUND; ARCA CAPITAL MANAGEMENT, LLC"
